#### Import DuckDB

In [1]:
import duckdb

#### Create the database

In [2]:
conn = duckdb.connect("insurance.duckdb")

#### Load CSV files into the database

In [3]:
conn.execute(""" 
CREATE OR REPLACE TABLE dim_adjusters AS 
SELECT *
FROM read_csv_auto('data/dim_adjusters.csv');
""")

conn.execute(""" 
CREATE OR REPLACE TABLE dim_customers AS 
SELECT *
FROM read_csv_auto('data/dim_customers.csv');
""")

conn.execute(""" 
CREATE OR REPLACE TABLE dim_policies AS 
SELECT *
FROM read_csv_auto('data/dim_policies.csv');
""")

conn.execute(""" 
CREATE OR REPLACE TABLE fact_claims AS 
SELECT *
FROM read_csv_auto('data/fact_claims.csv');
""")

#### Test and verify that the data was loaded in the database

In [15]:
from IPython.display import display, Markdown

display(Markdown("Check that the tables are in the DB"))
display( 
    conn.sql(""" 
    SHOW TABLES;
    """).df() 
)

display(Markdown("Quick data snapshot test for **dim_adjusters** table"))
display( 
    conn.sql(""" 
    SELECT * 
    FROM dim_adjusters 
    LIMIT 5;
    """).df()
)

display(Markdown("Quick data snapshot test for **dim_customers** table"))
display( 
    conn.sql(""" 
    SELECT * 
    FROM dim_customers
    LIMIT 5;
    """).df()
)

display(Markdown("Quick data snapshot test for **dim_policies** table"))
display( 
    conn.sql(""" 
    SELECT * 
    FROM dim_policies
    LIMIT 5;
    """).df()
)

display(Markdown("Quick data snapshot test for **fact_claims** table"))
display( 
    conn.sql(""" 
    SELECT * 
    FROM fact_claims
    LIMIT 5;
    """).df()
)

Check that the tables are in the DB

,name
0,dim_adjusters
1,dim_customers
2,dim_policies
3,fact_claims


Quick data snapshot test for **dim_adjusters** table

,adjuster_id,adjuster_name,region,hire_date
0,ADJ-001,J. Romero,Northeast,2017-03-06
1,ADJ-002,A. Singh,West,2015-06-09
2,ADJ-003,M. Chen,Midwest,2019-02-01
3,ADJ-004,L. Garcia,Midwest,2018-02-13
4,ADJ-005,K. Patel,Northeast,2012-08-08


Quick data snapshot test for **dim_customers** table

,customer_id,first_name,last_name,date_of_birth,email,signup_date
0,CUST-20001,Jose,Moore,1991-10-11,jose.moore1@mail.com,2017-02-16
1,CUST-20002,Sandra,Flores,1992-09-06,sandra.flores2@mail.com,2016-01-21
2,CUST-20003,Jose,Rodriguez,1962-09-21,jose.rodriguez3@mail.com,2022-10-11
3,CUST-20004,Maria,Reyes,1985-07-17,maria.reyes4@mail.com,2021-11-26
4,CUST-20005,Sandra,Reyes,1951-05-07,sandra.reyes5@mail.com,2015-05-27


Quick data snapshot test for **dim_policies** table

,policy_id,customer_id,policy_type,state,coverage_limit,premium_amount,effective_date,expiration_date
0,POL-100001,CUST-20171,AUTO,IL,30322.92,320.95,2022-04-22,2023-04-22
1,POL-100002,CUST-20026,Renters,GA,40109.20,1273.39,2023-04-30,2024-04-29
2,POL-100003,CUST-20223,Home,wa,373132.34,9165.18,2022-12-02,2023-12-02
3,POL-100004,CUST-20441,Renters,GA,10830.75,266.67,2023-10-17,2024-10-16
4,POL-100005,CUST-20470,Health,TX,47381.30,1683.17,2022-03-22,2023-03-22


Quick data snapshot test for **fact_claims** table

,claim_id,policy_id,adjuster_id,claim_date,cause_of_loss,claim_amount,claim_status
0,CLM-10692,POL-100677,ADJ-007,2023-10-29,Collision,14985.34,In Review
1,CLM-10947,POL-100044,ADJ-010,2023-12-22,Emergency Visit,16307.17,Closed
2,CLM-11008,POL-100392,ADJ-008,2023-05-01,Outpatient Procedure,10643.69,Denied
3,CLM-10810,POL-100376,ADJ-002,2023-09-16,Fire,23605.88,Denied
4,CLM-11370,POL-100715,ADJ-008,2023-03-27,Vandalism,13189.58,Closed


## SQL TECHNICAL INTERVIEW PREP

### SECTION A - Joins Accross Multiple Tables

#### Q1 — JOIN

**Write a query that returns each claim with the customer's full name (`first_name + last_name`), the policy type, and the claim amount. Only include claims where `claim_amount` is greater than 5,000.**

> 💡 **Hint:** You'll need to join `fact_claims → dim_policies → dim_customers`. Think about which join type to use and why.


In [19]:
conn.sql(""" 
    SELECT cl.claim_id,
           cu.first_name,
           cu.last_name,
           po.policy_type,
           cl.claim_amount
             
    FROM fact_claims cl
    JOIN dim_policies po ON cl.policy_id = po.policy_id
    JOIN dim_customers cu ON po.customer_id = cu.customer_id 
    WHERE cl.claim_amount > 5000 
    ORDER BY cl.claim_amount DESC    
    LIMIT 5;
    """).df()

,claim_id,first_name,last_name,policy_type,claim_amount
0,CLM-10780,Andres,Davis,Life,1354169.90
1,CLM-10664,Jose,Martinez,Home,1060810.63
2,CLM-10314,Carlos,Smith,Life,1053786.91
3,CLM-10292,Linda,Miller,life,931815.80
4,CLM-11147,Jose,Sanchez,Life,876942.69
